In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
import os
os.listdir('/content/drive/MyDrive')

['Colab Notebooks',
 'Untitled folder',
 'Gg',
 'A hyper-realistic cinematic portrait in ultra HD, ....gdoc',
 '1000041850.mp4',
 'Record_2025-11-16-16-48-25_6012fa4d4ddec268fc5c7112cbb265e7.mp4',
 'Parallel vs. Distributed Systems: A Comparison.gsheet',
 'OM_PRAKASH_SINGH_FlowCV_Resume_2026-03-03.pdf',
 'OM PRAKASH aadhar .pdf',
 'On',
 'Om Prakash Singh ',
 'Enhance this portrait while strictly preserving th....gdoc',
 'True.csv',
 'Fake.csv',
 'models',
 'fake_news_complete_save.zip',
 'FakeNewsProject']

In [35]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/fake_news_complete_save.zip"
extract_path = "/content/fake_news_complete_save"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully")
print(os.listdir(extract_path))

✅ Extracted successfully
['bert_finetuned', 'confusion_matrices.png', 'tfidf_vectorizer.pkl', 'thresholds.pkl', 'roc_curves.png', 'pr_curves.png', 'prob_distributions.png', 'model_lr.pkl', 'evaluation_dashboard.png', 'calibration_curves.png', 'threshold_sweep.png', 'metrics_heatmap.png', 'model_meta_stack.pkl', 'metrics_summary.csv', 'model_rf.pkl', 'model_metadata.json', 'model_svm.pkl']


In [36]:
!pip install -q transformers torch scikit-learn

In [37]:
import pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [41]:
BASE_PATH = "/content/fake_news_complete_save"
print("Using folder:", BASE_PATH)

Using folder: /content/fake_news_complete_save


In [42]:
!pip install joblib

In [43]:
import joblib

lr_model = joblib.load(f"{BASE_PATH}/model_lr.pkl")

print("✅ LR model loaded")
print(type(lr_model))

✅ LR model loaded
<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [44]:
rf_model = joblib.load(f"{BASE_PATH}/model_rf.pkl")

print("✅ RF model loaded")
print(type(rf_model))

✅ RF model loaded
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [45]:
svm_model = joblib.load(f"{BASE_PATH}/model_svm.pkl")

print("✅ SVM model loaded")
print(type(svm_model))

✅ SVM model loaded
<class 'sklearn.calibration.CalibratedClassifierCV'>


In [46]:
meta_model = joblib.load(f"{BASE_PATH}/model_meta_stack.pkl")

print("✅ Meta model loaded")
print(type(meta_model))
print(meta_model)

✅ Meta model loaded
<class 'sklearn.linear_model._logistic.LogisticRegression'>
LogisticRegression(class_weight='balanced', max_iter=1000)


In [47]:
vectorizer = joblib.load(f"{BASE_PATH}/tfidf_vectorizer.pkl")

print("✅ Vectorizer loaded")
print(type(vectorizer))

✅ Vectorizer loaded
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [48]:
print(meta_model)

LogisticRegression(class_weight='balanced', max_iter=1000)


In [49]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

bert_path = "/content/fake_news_complete_save/bert_finetuned"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_path)
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)
bert_model.eval()

print("✅ BERT loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ BERT loaded successfully


In [50]:
def predict_both_models(text):
    # -------- STACKING PART --------
    X = vectorizer.transform([text])

    lr_prob = lr_model.predict_proba(X)[0][1]
    rf_prob = rf_model.predict_proba(X)[0][1]
    svm_prob = svm_model.predict_proba(X)[0][1]

    meta_features = np.array([[lr_prob, rf_prob, svm_prob]])

    stack_prob = meta_model.predict_proba(meta_features)[0][1]
    stack_pred = meta_model.predict(meta_features)[0]
    stack_label = "REAL" if stack_pred == 1 else "FAKE"

    # -------- BERT PART --------
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    bert_pred = int(np.argmax(probs))
    bert_prob = float(probs[1])
    bert_label = "REAL" if bert_pred == 1 else "FAKE"

    # -------- OUTPUT --------
    print("\n==============================")
    print("INPUT NEWS:")
    print(text)
    print("==============================")

    print("\nSTACKING MODEL RESULT")
    print("Prediction :", stack_label)
    print("Confidence :", round(stack_prob, 4))

    print("\nBERT MODEL RESULT")
    print("Prediction :", bert_label)
    print("Confidence :", round(bert_prob, 4))

In [51]:
predict_both_models("Government announces new policy")
predict_both_models("Aliens found in earth core")


INPUT NEWS:
Government announces new policy

STACKING MODEL RESULT
Prediction : REAL
Confidence : 0.8338

BERT MODEL RESULT
Prediction : FAKE
Confidence : 0.0218

INPUT NEWS:
Aliens found in earth core

STACKING MODEL RESULT
Prediction : REAL
Confidence : 0.8598

BERT MODEL RESULT
Prediction : REAL
Confidence : 0.813


In [52]:
def predict_both_models(text):
    # -------- STACKING --------
    X = vectorizer.transform([text])

    lr_prob = lr_model.predict_proba(X)[0][1]
    rf_prob = rf_model.predict_proba(X)[0][1]
    svm_prob = svm_model.predict_proba(X)[0][1]

    meta_features = np.array([[lr_prob, rf_prob, svm_prob]])

    stack_prob_real = meta_model.predict_proba(meta_features)[0][1]
    stack_pred = meta_model.predict(meta_features)[0]
    stack_label = "REAL" if stack_pred == 1 else "FAKE"
    stack_conf = stack_prob_real if stack_label == "REAL" else 1 - stack_prob_real

    # -------- BERT --------
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    bert_pred = int(np.argmax(probs))
    bert_prob_real = float(probs[1])
    bert_label = "REAL" if bert_pred == 1 else "FAKE"
    bert_conf = bert_prob_real if bert_label == "REAL" else 1 - bert_prob_real

    # -------- PROFESSIONAL OUTPUT --------
    print("\n" + "="*50)
    print("📰 FAKE NEWS DETECTION SYSTEM")
    print("="*50)

    print("\n📥 INPUT NEWS:")
    print(text[:300] + "..." if len(text) > 300 else text)

    print("\n" + "-"*50)
    print("📊 MODEL PREDICTIONS")
    print("-"*50)

    # Stacking
    print("\n🔹 Stacking Model (LR + RF + SVM)")
    print(f"   Prediction  : {stack_label}")
    print(f"   Confidence  : {stack_conf:.4f}")

    # BERT
    print("\n🔹 BERT Model (Deep Learning)")
    print(f"   Prediction  : {bert_label}")
    print(f"   Confidence  : {bert_conf:.4f}")

    print("\n" + "-"*50)
    print("📈 Base Model Probabilities")
    print("-"*50)
    print(f"   Logistic Regression : {lr_prob:.4f}")
    print(f"   Random Forest       : {rf_prob:.4f}")
    print(f"   SVM                 : {svm_prob:.4f}")

    print("="*50 + "\n")

In [53]:
print("\n🔍 Enter News for Detection\n")
news = input("📝 Your Input: ")
predict_both_models(news)


🔍 Enter News for Detection

📝 Your Input: fff

📰 FAKE NEWS DETECTION SYSTEM

📥 INPUT NEWS:
fff

--------------------------------------------------
📊 MODEL PREDICTIONS
--------------------------------------------------

🔹 Stacking Model (LR + RF + SVM)
   Prediction  : REAL
   Confidence  : 0.9758

🔹 BERT Model (Deep Learning)
   Prediction  : FAKE
   Confidence  : 0.8658

--------------------------------------------------
📈 Base Model Probabilities
--------------------------------------------------
   Logistic Regression : 0.9496
   Random Forest       : 0.9529
   SVM                 : 0.9560

